In [ ]:
import os
import json
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from google.colab import drive
import zipfile

# Mount Google Drive
drive.mount('/content/drive')
OUTPUT_PATH = "/content/drive/MyDrive/Tugas SMT 6/Propen/newforgait_f_dataset_cva_hs.pkl"

with open(OUTPUT_PATH, "rb") as f:
    dataset = pickle.load(f)

print("Number of trials loaded:", len(dataset))
print("Number of samples in first trial's sensor matrix:", dataset[0]["sensor_matrix"].shape[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of trials loaded: 488
Number of samples in first trial's sensor matrix: 3639


Data Preprocessing

In [ ]:
import numpy as np

# =========================================================
# WINDOWING SENSOR DATA
# =========================================================
def preprocess_sensor_data(sensor_matrix, window_size=100, stride=50):
    """
    Convert continuous sensor data into sliding windows.

    Output:
    (n_windows, window_size, n_features)
    """

    sequences = []

    n_samples = len(sensor_matrix)

    for start_idx in range(0, n_samples - window_size + 1, stride):

        end_idx = start_idx + window_size

        window = sensor_matrix[start_idx:end_idx]

        sequences.append(window)

    return np.array(sequences, dtype=np.float32)


# =========================================================
# CREATE SEQUENCE LABELS
# =========================================================
def create_sequence_labels(
    gait_events,
    data_length,
    window_size=100,
    stride=50
):
    """
    Create per-timestep labels for gait events.

    Output:
    (n_windows, window_size)

    Label:
    0 = non-event
    1 = gait-event region
    """

    labels = []

    n_windows = len(range(0, data_length - window_size + 1, stride))

    for w_idx in range(n_windows):

        window_start = w_idx * stride
        window_end = window_start + window_size

        # Label untuk tiap timestep dalam window
        window_label = np.zeros(window_size, dtype=np.int32)

        # Cek semua gait event
        for event in gait_events:

            # Safety check
            if len(event) < 2:
                continue

            event_start, event_end = event

            # Cari overlap
            overlap_start = max(window_start, event_start)
            overlap_end = min(window_end, event_end)

            # Jika overlap ada
            if overlap_start < overlap_end:

                # Konversi ke index lokal window
                local_start = overlap_start - window_start
                local_end = overlap_end - window_start

                # Isi region event dengan label 1
                window_label[local_start:local_end] = 1

        labels.append(window_label)

    return np.array(labels, dtype=np.int32)


# =========================================================
# PREPROCESS ALL DATASET
# =========================================================

X_data = []

y_left = []
y_right = []

fma_scores = []
meta_features_list = []
subject_ids = []

WINDOW_SIZE = 100
STRIDE = 50

print("====================================")
print("Preprocessing dataset...")
print("====================================")

for trial_idx, trial in enumerate(dataset):

    try:

        # =====================================================
        # LOAD DATA
        # =====================================================
        sensor_matrix = trial["sensor_matrix"]

        left_events = trial["left_gait_events"]
        right_events = trial["right_gait_events"]

        fma_score = trial["fma_score"]

        meta_features = trial["meta_features"]

        subject_id = trial["subject_id"]

        # =====================================================
        # SKIP SHORT DATA
        # =====================================================
        if len(sensor_matrix) < WINDOW_SIZE:
            continue

        # =====================================================
        # WINDOWING
        # =====================================================
        sequences = preprocess_sensor_data(
            sensor_matrix,
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        # =====================================================
        # LABELING
        # =====================================================
        left_labels = create_sequence_labels(
            left_events,
            len(sensor_matrix),
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        right_labels = create_sequence_labels(
            right_events,
            len(sensor_matrix),
            window_size=WINDOW_SIZE,
            stride=STRIDE
        )

        # =====================================================
        # SAFETY CHECK
        # =====================================================
        if len(sequences) != len(left_labels):

            print(f"Mismatch at trial {trial_idx}")

            print(f"Sequences: {len(sequences)}")
            print(f"Labels   : {len(left_labels)}")

            continue

        # =====================================================
        # APPEND DATA
        # =====================================================
        X_data.extend(sequences)

        y_left.extend(left_labels)
        y_right.extend(right_labels)

        # Metadata per sequence
        for _ in range(len(sequences)):

            fma_scores.append(fma_score)

            meta_features_list.append(meta_features)

            subject_ids.append(subject_id)

    except Exception as e:

        print(f"Error at trial {trial_idx}: {e}")

        continue


# =========================================================
# CONVERT TO NUMPY
# =========================================================
X_data = np.array(X_data, dtype=np.float32)

y_left = np.array(y_left, dtype=np.int32)
y_right = np.array(y_right, dtype=np.int32)

fma_scores = np.array(fma_scores, dtype=np.float32)

subject_ids = np.array(subject_ids)

# =========================================================
# SUMMARY
# =========================================================
print("\n====================================")
print("PREPROCESSING FINISHED")
print("====================================")

print(f"Total sequences : {len(X_data)}")

print(f"\nX_data shape    : {X_data.shape}")
print(f"y_left shape    : {y_left.shape}")
print(f"y_right shape   : {y_right.shape}")

print(f"FMA shape       : {fma_scores.shape}")

# =========================================================
# LABEL STATISTICS
# =========================================================

print("\nLeft label statistics:")
print("Total event frames     :", np.sum(y_left == 1))
print("Total non-event frames :", np.sum(y_left == 0))

print("\nRight label statistics:")
print("Total event frames     :", np.sum(y_right == 1))
print("Total non-event frames :", np.sum(y_right == 0))

print("====================================")

Preprocessing dataset...

PREPROCESSING FINISHED
Total sequences : 25058

X_data shape    : (25058, 100, 13)
y_left shape    : (25058, 100)
y_right shape   : (25058, 100)
FMA shape       : (25058,)

Left label statistics:
Total event frames     : 741031
Total non-event frames : 1764769

Right label statistics:
Total event frames     : 733831
Total non-event frames : 1771969


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# =========================================================
# UNIQUE SUBJECTS
# =========================================================
unique_subjects = np.unique(subject_ids)

print("====================================")
print("SUBJECT SPLITTING")
print("====================================")

print("Total unique subjects :", len(unique_subjects))

# =========================================================
# SUBJECT-INDEPENDENT SPLIT
# =========================================================
train_subjects, test_subjects = train_test_split(
    unique_subjects,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train subjects :", len(train_subjects))
print("Test subjects  :", len(test_subjects))

# =========================================================
# CREATE MASK
# =========================================================
train_mask = np.isin(subject_ids, train_subjects)
test_mask = np.isin(subject_ids, test_subjects)

# =========================================================
# SPLIT INPUT
# =========================================================
X_train = X_data[train_mask]
X_test = X_data[test_mask]

# =========================================================
# SPLIT LABELS
# =========================================================
y_left_train = y_left[train_mask]
y_left_test = y_left[test_mask]

y_right_train = y_right[train_mask]
y_right_test = y_right[test_mask]

# =========================================================
# OPTIONAL AUXILIARY DATA
# =========================================================
fma_train = fma_scores[train_mask]
fma_test = fma_scores[test_mask]

subject_train = subject_ids[train_mask]
subject_test = subject_ids[test_mask]

# =========================================================
# SUMMARY
# =========================================================
print("\n====================================")
print("TRAIN TEST SPLIT FINISHED")
print("====================================")

print("\nTRAIN DATA")
print("X_train shape       :", X_train.shape)
print("y_left_train shape  :", y_left_train.shape)
print("y_right_train shape :", y_right_train.shape)

print("\nTEST DATA")
print("X_test shape        :", X_test.shape)
print("y_left_test shape   :", y_left_test.shape)
print("y_right_test shape  :", y_right_test.shape)

# =========================================================
# LABEL DISTRIBUTION
# =========================================================

# LEFT TRAIN
left_train_event = np.sum(y_left_train == 1)
left_train_nonevent = np.sum(y_left_train == 0)

# LEFT TEST
left_test_event = np.sum(y_left_test == 1)
left_test_nonevent = np.sum(y_left_test == 0)

# RIGHT TRAIN
right_train_event = np.sum(y_right_train == 1)
right_train_nonevent = np.sum(y_right_train == 0)

# RIGHT TEST
right_test_event = np.sum(y_right_test == 1)
right_test_nonevent = np.sum(y_right_test == 0)

print("\n====================================")
print("LABEL DISTRIBUTION")
print("====================================")

print("\nLEFT TRAIN")
print("Event frames     :", left_train_event)
print("Non-event frames :", left_train_nonevent)

print("\nLEFT TEST")
print("Event frames     :", left_test_event)
print("Non-event frames :", left_test_nonevent)

print("\nRIGHT TRAIN")
print("Event frames     :", right_train_event)
print("Non-event frames :", right_train_nonevent)

print("\nRIGHT TEST")
print("Event frames     :", right_test_event)
print("Non-event frames :", right_test_nonevent)

# =========================================================
# SUBJECT LEAKAGE CHECK
# =========================================================
overlap_subjects = set(train_subjects).intersection(set(test_subjects))

print("\n====================================")
print("LEAKAGE CHECK")
print("====================================")

if len(overlap_subjects) == 0:
    print("No subject leakage detected")
else:
    print("WARNING: Subject leakage detected!")
    print(overlap_subjects)

print("====================================")

SUBJECT SPLITTING
Total unique subjects : 122
Train subjects : 97
Test subjects  : 25

TRAIN TEST SPLIT FINISHED

TRAIN DATA
X_train shape       : (20333, 100, 13)
y_left_train shape  : (20333, 100)
y_right_train shape : (20333, 100)

TEST DATA
X_test shape        : (4725, 100, 13)
y_left_test shape   : (4725, 100)
y_right_test shape  : (4725, 100)

LABEL DISTRIBUTION

LEFT TRAIN
Event frames     : 594654
Non-event frames : 1438646

LEFT TEST
Event frames     : 146377
Non-event frames : 326123

RIGHT TRAIN
Event frames     : 596458
Non-event frames : 1436842

RIGHT TEST
Event frames     : 137373
Non-event frames : 335127

LEAKAGE CHECK
No subject leakage detected


##Build Model

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

# SENSOR INPUT
input_sensor = Input(shape=(100, 13))

# CNN FEATURE EXTRACTION
x = Conv1D(
    filters=32,
    kernel_size=5,
    padding='same',
    activation='relu'
)(input_sensor)

x = BatchNormalization()(x)
x = Conv1D(
    filters=64,
    kernel_size=3,
    padding='same',
    activation='relu'
)(x)

x = BatchNormalization()(x)
x = Dropout(0.2)(x)

# BIGRU TEMPORAL MODELING
x = Bidirectional(
    GRU(
        64,
        return_sequences=True
    )
)(x)

x = Dropout(0.2)(x)
x = Bidirectional(
    GRU(
        64,
        return_sequences=True
    )
)(x)

# ATTENTION
attention = Attention()([x, x])
x = concatenate([x, attention])

# TEMPORAL DENSE
x = TimeDistributed(
    Dense(64, activation='relu')
)(x)
x = Dropout(0.2)(x)

# OUTPUT
output = TimeDistributed(
    Dense(1, activation='sigmoid')
)(x)

# BUILD MODEL
model = Model(
    inputs=input_sensor,
    outputs=output
)

# COMPILE
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

# SUMMARY
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100, 13)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 32)   │      2,112 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 100, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 100, 64)   │      6,208 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 128)  │     49,920 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 100, 128)  │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 100, 128)  │     74,496 │ dropout_1[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 100, 128)  │          0 │ bidirectional_1[… │
│ (Attention)         │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100, 256)  │          0 │ bidirectional_1[… │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 100, 64)   │     16,448 │ concatenate[0][0] │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 100, 64)   │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 100, 1)    │         65 │ dropout_2[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 149,633 (584.50 KB)

 Trainable params: 149,441 (583.75 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
print(y_left_train.shape)
print(y_left_test.shape)

(20333, 100)
(4725, 100)


In [ ]:
print(np.isnan(X_train).sum())
print(np.isinf(X_train).sum())

0
0


In [ ]:
# =========================================================
# CEK 1 SAMPLE
# =========================================================

sample = dataset[0]

print("\n====================================")
print("SAMPLE STRUCTURE")
print("====================================")

print("Keys:")
print(sample.keys())

# =========================================================
# SENSOR DATA
# =========================================================

sensor_matrix = sample["sensor_matrix"]

print("\n====================================")
print("SENSOR MATRIX")
print("====================================")

print("Shape :", sensor_matrix.shape)
print("Dtype :", sensor_matrix.dtype)

print("\nFirst 5 rows:")
print(sensor_matrix[:5])

# =========================================================
# GROUND TRUTH
# =========================================================

print("\n====================================")
print("LEFT GAIT EVENTS")
print("====================================")

print(sample["left_gait_events"][:5])

print("\n====================================")
print("RIGHT GAIT EVENTS")
print("====================================")

print(sample["right_gait_events"][:5])

# =========================================================
# FMA SCORE
# =========================================================

print("\n====================================")
print("FMA SCORE")
print("====================================")

print(sample["fma_score"])

# =========================================================
# META FEATURES
# =========================================================

print("\n====================================")
print("META FEATURES")
print("====================================")

for key, value in sample["meta_features"].items():
    print(f"{key}: {value}")

# =========================================================
# SUBJECT ID
# =========================================================

print("\n====================================")
print("SUBJECT ID")
print("====================================")

print(sample["subject_id"])

# =========================================================
# DATASET STATISTICS
# =========================================================

print("\n====================================")
print("DATASET STATISTICS")
print("====================================")

all_lengths = []

total_left_events = 0
total_right_events = 0

nan_count = 0
inf_count = 0

for trial in dataset:

    sensor_matrix = trial["sensor_matrix"]

    all_lengths.append(len(sensor_matrix))

    total_left_events += len(trial["left_gait_events"])
    total_right_events += len(trial["right_gait_events"])

    nan_count += np.isnan(sensor_matrix).sum()
    inf_count += np.isinf(sensor_matrix).sum()

print(f"Average signal length : {np.mean(all_lengths):.2f}")
print(f"Min signal length     : {np.min(all_lengths)}")
print(f"Max signal length     : {np.max(all_lengths)}")

print(f"\nTotal left events     : {total_left_events}")
print(f"Total right events    : {total_right_events}")

print(f"\nTotal NaN values      : {nan_count}")
print(f"Total Inf values      : {inf_count}")

print("====================================")


SAMPLE STRUCTURE
Keys:
dict_keys(['subject_id', 'sensor_matrix', 'fma_score', 'left_gait_events', 'right_gait_events', 'meta_features'])

SENSOR MATRIX
Shape : (3639, 13)
Dtype : float32

First 5 rows:
[[ 5.9435200e-03 -1.0114911e-02 -5.5487771e-03  6.5967422e-03
   3.6224558e-03  1.6884865e-03 -1.1915779e-02  1.7015599e-02
  -2.2389803e-02  4.1068653e-03 -7.5857462e-03  1.0852003e-03
   0.0000000e+00]
 [ 9.8384144e-06 -5.5660023e-03 -8.7528452e-03  2.9613583e-03
   4.5869686e-03  3.6643883e-03 -1.5090008e-02  7.1444544e-03
  -5.2312226e-03  3.0839193e-04 -1.3503063e-03  2.0316863e-03
   0.0000000e+00]
 [-4.8699188e-03 -1.5221416e-03 -9.9367779e-03 -1.4641513e-04
   4.7878935e-03  4.9970876e-03 -1.6797783e-02  2.8762969e-04
   7.8627504e-03 -2.6697754e-03  2.5777696e-03  2.6165647e-03
   0.0000000e+00]
 [-7.9993661e-03  1.7054523e-03 -8.1829503e-03 -2.3702912e-03
   3.8589099e-03  5.2937479e-03 -1.6317943e-02 -2.1133160e-03
   1.4631365e-02 -4.1673551e-03  3.0963740e-03  2.6636219e-03

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

# RESHAPE LABEL
y_left_train = y_left_train[..., np.newaxis]
y_left_test = y_left_test[..., np.newaxis]
y_right_train = y_right_train[..., np.newaxis]
y_right_test = y_right_test[..., np.newaxis]

# CALLBACKS
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# Save best model
checkpoint = ModelCheckpoint(
    filepath='best_gait_event_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Reduce learning rate otomatis
optimizer=tf.keras.optimizers.Adam(
    learning_rate=1e-4,
    clipnorm=1.0
)

# TRAIN MODEL
history = model.fit(
    X_train,
    y_left_train,
    validation_data=(
        X_test,
        y_left_test
    ),
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stop,
        checkpoint,
        reduce_lr
    ],
    verbose=1
)

# SAVE FINAL MODEL
model.save("final_gait_event_model.keras")

print("====================================")
print("TRAINING FINISHED")
print("Best model saved:")
print("best_gait_event_model.keras")
print("====================================")

NameError: name 'reduce_lr' is not defined

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

# =========================================================
# SAFETY CLEANING
# =========================================================

X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)

# =========================================================
# RESHAPE LABEL
# =========================================================
# Dari:
# (batch,100)
#
# Menjadi:
# (batch,100,1)

if len(y_left_train.shape) == 2:
    y_left_train = y_left_train[..., np.newaxis]

if len(y_left_test.shape) == 2:
    y_left_test = y_left_test[..., np.newaxis]

if len(y_right_train.shape) == 2:
    y_right_train = y_right_train[..., np.newaxis]

if len(y_right_test.shape) == 2:
    y_right_test = y_right_test[..., np.newaxis]

# =========================================================
# COMPILE MODEL
# =========================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),

    loss='binary_crossentropy',

    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

# =========================================================
# CALLBACKS
# =========================================================

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# Save best model
checkpoint = ModelCheckpoint(
    filepath='best_gait_event_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Reduce LR otomatis
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# =========================================================
# TRAIN MODEL
# =========================================================

history = model.fit(

    X_train,
    y_left_train,

    validation_data=(
        X_test,
        y_left_test
    ),

    epochs=100,

    batch_size=32,

    callbacks=[
        early_stop,
        checkpoint,
        reduce_lr
    ],
    verbose=1
)

# =========================================================
# SAVE FINAL MODEL
# =========================================================

model.save("final_gait_event_model.keras")

print("====================================")
print("TRAINING FINISHED")
print("Best model saved:")
print("best_gait_event_model.keras")
print("====================================")

Epoch 1/100
636/636 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9020 - loss: 0.2311 - precision_1: 0.8348 - recall_1: 0.8476
Epoch 1: val_loss improved from None to 0.10474, saving model to best_gait_event_model.keras

Epoch 1: finished saving model to best_gait_event_model.keras
636/636 ━━━━━━━━━━━━━━━━━━━━ 53s 54ms/step - accuracy: 0.9549 - loss: 0.1202 - precision_1: 0.9227 - recall_1: 0.9230 - val_accuracy: 0.9601 - val_loss: 0.1047 - val_precision_1: 0.9338 - val_recall_1: 0.9378 - learning_rate: 1.0000e-04
Epoch 2/100
635/636 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9769 - loss: 0.0601 - precision_1: 0.9580 - recall_1: 0.9635
Epoch 2: val_loss improved from 0.10474 to 0.09998, saving model to best_gait_event_model.keras

Epoch 2: finished saving model to best_gait_event_model.keras
636/636 ━━━━━━━━━━━━━━━━━━━━ 30s 47ms/step - accuracy: 0.9780 - loss: 0.0573 - precision_1: 0.9600 - recall_1: 0.9649 - val_accuracy: 0.9622 - val_loss: 0.1000 - val_precision_1: 0.9466 - va

In [ ]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    jaccard_score
)

# =========================================================
# PREDICT
# =========================================================

y_pred_prob = model.predict(X_test)

# =========================================================
# THRESHOLDING
# =========================================================

threshold = 0.5

y_pred = (y_pred_prob > threshold).astype(np.int32)

# =========================================================
# FLATTEN
# =========================================================
# Dari:
# (batch,100,1)
#
# Menjadi:
# (total_frames,)

y_true_flat = y_left_test.flatten()

y_pred_flat = y_pred.flatten()

# =========================================================
# METRICS
# =========================================================

accuracy = accuracy_score(
    y_true_flat,
    y_pred_flat
)

precision = precision_score(
    y_true_flat,
    y_pred_flat,
    zero_division=0
)

recall = recall_score(
    y_true_flat,
    y_pred_flat,
    zero_division=0
)

f1 = f1_score(
    y_true_flat,
    y_pred_flat,
    zero_division=0
)

iou = jaccard_score(
    y_true_flat,
    y_pred_flat,
    zero_division=0
)

# =========================================================
# PRINT RESULTS
# =========================================================

print("====================================")
print("FRAME-LEVEL EVALUATION")
print("====================================")

print(f"Accuracy : {accuracy:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"F1-Score : {f1:.4f}")

print(f"IoU Score: {iou:.4f}")

# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_true_flat,
    y_pred_flat
)

print("\n====================================")
print("CONFUSION MATRIX")
print("====================================")

print(cm)

# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print("\n====================================")
print("CLASSIFICATION REPORT")
print("====================================")

print(
    classification_report(
        y_true_flat,
        y_pred_flat,
        digits=4
    )
)

148/148 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step
FRAME-LEVEL EVALUATION
Accuracy : 0.9653
Precision: 0.9352
Recall   : 0.9540
F1-Score : 0.9445
IoU Score: 0.8949

CONFUSION MATRIX
[[316452   9671]
 [  6727 139650]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9792    0.9703    0.9747    326123
           1     0.9352    0.9540    0.9445    146377

    accuracy                         0.9653    472500
   macro avg     0.9572    0.9622    0.9596    472500
weighted avg     0.9656    0.9653    0.9654    472500



In [ ]:
import numpy as np
import pandas as pd

# =========================================================
# CONVERT BINARY MASK -> INTERVALS
# =========================================================

def mask_to_intervals(mask):

    intervals = []

    in_event = False

    start = 0

    for i, val in enumerate(mask):

        # mulai event
        if val == 1 and not in_event:

            start = i
            in_event = True

        # akhir event
        elif val == 0 and in_event:

            end = i - 1

            intervals.append([start, end])

            in_event = False

    # handle event sampai akhir
    if in_event:

        intervals.append([start, len(mask)-1])

    return intervals


# =========================================================
# PREDICT
# =========================================================

y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob > 0.5).astype(np.int32)

# =========================================================
# EVALUATE SAMPLE
# =========================================================

results = []

num_samples = min(20, len(y_left_test))

for sample_idx in range(num_samples):

    # =====================================================
    # GROUND TRUTH
    # =====================================================

    true_mask = y_left_test[sample_idx].squeeze()

    # =====================================================
    # PREDICTION
    # =====================================================

    pred_mask = y_pred[sample_idx].squeeze()

    # =====================================================
    # CONVERT TO INTERVALS
    # =====================================================

    true_intervals = mask_to_intervals(true_mask)

    pred_intervals = mask_to_intervals(pred_mask)

    # =====================================================
    # MATCH EVENTS
    # =====================================================

    n_pairs = min(len(true_intervals), len(pred_intervals))

    for i in range(n_pairs):

        true_start, true_end = true_intervals[i]

        pred_start, pred_end = pred_intervals[i]

        # Error
        onset_error = pred_start - true_start

        offset_error = pred_end - true_end

        duration_true = true_end - true_start

        duration_pred = pred_end - pred_start

        duration_error = duration_pred - duration_true

        results.append({

            "sample": sample_idx,

            "true_interval": [true_start, true_end],

            "pred_interval": [pred_start, pred_end],

            "onset_error": onset_error,

            "offset_error": offset_error,

            "duration_error": duration_error
        })

# =========================================================
# DATAFRAME
# =========================================================

results_df = pd.DataFrame(results)

# =========================================================
# SHOW RESULTS
# =========================================================

print("====================================")
print("ACTUAL VS PREDICTED EVENTS")
print("====================================")

print(results_df.head(20))

# =========================================================
# ERROR STATISTICS
# =========================================================

print("\n====================================")
print("TEMPORAL ERROR STATISTICS")
print("====================================")

print(
    f"Mean onset error    : "
    f"{results_df['onset_error'].abs().mean():.2f} frames"
)

print(
    f"Mean offset error   : "
    f"{results_df['offset_error'].abs().mean():.2f} frames"
)

print(
    f"Mean duration error : "
    f"{results_df['duration_error'].abs().mean():.2f} frames"
)

148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step
ACTUAL VS PREDICTED EVENTS
    sample true_interval pred_interval  onset_error  offset_error  \
0        8      [87, 99]      [86, 99]           -1             0   
1        9      [37, 58]      [36, 66]           -1             8   
2       10        [0, 8]       [0, 15]            0             7   
3       11      [70, 99]      [71, 99]            1             0   
4       12      [20, 65]      [21, 64]            1            -1   
5       13       [0, 15]       [0, 14]            0            -1   
6       14      [60, 99]      [61, 99]            1             0   
7       15      [10, 54]      [11, 50]            1            -4   
8       16        [0, 4]      [96, 99]           96            95   
9       17      [45, 87]      [46, 86]            1            -1   
10      18       [0, 37]       [0, 36]            0            -1   
11      19      [88, 99]      [89, 99]            1             0   

    duration_error  
0           